# Credit Card Fraud Detection — Detecting Fraudulent Transactions

**Data:** Kaggle Credit Card Fraud Detection Dataset — 284,807 transactions, 28 PCA features, 0.17% fraud rate

**Problem:** Given 28 PCA-transformed features (V1–V28), transaction amount, and timestamp, predict whether a transaction is fraudulent (Class=1) or legitimate (Class=0).

**TL;DR:** I built a model that predicts fraudulent credit card transactions from 28 PCA-transformed features using Logistic Regression, Random Forest, and XGBoost. The best models (XGBoost and Random Forest) achieve a PR-AUC of ~0.82 — catching approximately 75-80% of fraud cases with 91-96% precision. Tree-based models significantly outperform logistic regression (PR-AUC 0.82 vs 0.68), showing fraud involves non-linear feature interactions. V14 is the most important feature for detecting fraud, followed by V17 and V10.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix, precision_recall_curve,
    average_precision_score, precision_score,
    recall_score, f1_score
)
import xgboost as xgb
import os.path

sns.set_style("whitegrid")
print("All imports loaded successfully")

In [ ]:
if not os.path.exists("data.csv"):
    print("=" * 60)
    print("DATA NOT FOUND - Download from Kaggle:")
    print("  https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud")
    print("Place as 'data.csv' in this folder, then re-run.")
    print("=" * 60)
    raise FileNotFoundError("data.csv not found")

df = pd.read_csv("data.csv")
print(f"Dataset: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

In [ ]:
df.head(10)

In [ ]:
# Data info and summary
print("=== Data Types and Non-Null Counts ===")
df.info(verbose=False)
print(f"\n=== Shape: {df.shape} ===")
print(f"\n=== Summary Statistics ===")
df.describe()

In [ ]:
# Check for missing values
missing = df.isnull().sum().sum()
print(f"Total missing values: {missing}")
if missing == 0:
    print("No missing values found")

## Data Cleaning

In [ ]:
# Remove duplicate rows
orig_len = len(df)
dup_count = df.duplicated().sum()
print(f"Duplicate rows: {dup_count:,} ({dup_count/orig_len*100:.2f}%)")
df = df.drop_duplicates()
print(f"After removal: {df.shape[0]:,} rows ({df.shape[0]/orig_len*100:.1f}% of original)")

In [ ]:
# Verify data integrity after cleaning
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"\nClass distribution:")
class_counts = df["Class"].value_counts().sort_index()
for cls, count in class_counts.items():
    label = "Fraud" if cls == 1 else "Legitimate"
    print(f"  {label} ({cls}): {count:>6,} ({count/len(df)*100:.4f}%)")
print(f"\nFraud rate: {df['Class'].mean()*100:.4f}%")
print(f"Class imbalance ratio: 1:{df['Class'].value_counts()[0] // df['Class'].value_counts().get(1, 1)}")

## Exploratory Analysis

In [ ]:
# Chart 1: Class Distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
colors = ["#2196F3", "#FF9800"]  # blue/orange: colorblind-safe
labels = ["Legitimate (0)", "Fraud (1)"]
counts = df["Class"].value_counts().sort_index()
pcts = counts / len(df) * 100

bars1 = ax1.bar(labels, counts.values, color=colors, edgecolor="white", width=0.6)
ax1.set_ylabel("Number of Transactions")
ax1.set_title("Transaction Counts")
for bar, val in zip(bars1, counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(counts)*0.02,
             f"{val:,}", ha="center", va="bottom", fontweight="bold")

bars2 = ax2.bar(labels, pcts.values, color=colors, edgecolor="white", width=0.6)
ax2.set_ylabel("Percentage (%)")
ax2.set_title("Class Distribution (%)")
for bar, val in zip(bars2, pcts.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f"{val:.4f}%", ha="center", va="bottom", fontweight="bold")

fig.suptitle("Target Class Distribution", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()
print("Only 0.17% of transactions are fraudulent — extreme class imbalance.")

In [ ]:
# Chart 2: Top Features Correlated with Fraud
corr_all = df.corr()["Class"].drop("Class")
top_idx = corr_all.abs().sort_values(ascending=False).head(10).index
top_corr = corr_all[top_idx].sort_values()

fig, ax = plt.subplots(figsize=(10, 5))
colors_bar = ["#FF9800" if v < 0 else "#2196F3" for v in top_corr.values]
ax.barh(range(len(top_corr)), top_corr.values, color=colors_bar, edgecolor="white", height=0.6)
ax.set_yticks(range(len(top_corr)))
ax.set_yticklabels(top_corr.index, fontsize=10)
ax.set_xlabel("Pearson Correlation with Class (Fraud)", fontsize=11)
ax.set_title("Top 10 Features by Correlation with Fraud", fontsize=13, fontweight="bold")
ax.axvline(x=0, color="gray", linestyle="-", linewidth=0.5, alpha=0.5)
ax.grid(True, axis="x", alpha=0.3)

for idx, val in enumerate(top_corr.values):
    label = f"r = {val:+.4f}"
    if val < 0:
        ax.text(val - 0.005, idx, label, ha="right", va="center", fontsize=8)
    else:
        ax.text(val + 0.005, idx, label, ha="left", va="center", fontsize=8)

plt.tight_layout()
plt.show()

print("V14, V17, V10, and V12 have the strongest correlation with fraud.")
print("Negative correlation means lower feature values are associated with fraud, and vice versa.")

In [ ]:
# Chart 3: Amount Analysis by Class
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

fraud_amounts = df[df["Class"] == 1]["Amount"]
legit_amounts = df[df["Class"] == 0]["Amount"]

# Histogram (clipped to $500 so bars are readable)
axes[0].hist(legit_amounts.clip(upper=500), bins=50, alpha=0.5, color="#2196F3", label="Legitimate", density=True)
axes[0].hist(fraud_amounts.clip(upper=500), bins=50, alpha=0.7, color="#FF9800", label="Fraud", density=True)
axes[0].set_xlabel("Transaction Amount ($)")
axes[0].set_ylabel("Density")
axes[0].set_title("Amount Distribution (≤$500)")
axes[0].legend()

# Box plot
bp = axes[1].boxplot([legit_amounts, fraud_amounts], patch_artist=True, showfliers=False, widths=0.5)
bp["boxes"][0].set_facecolor("#2196F3")
bp["boxes"][1].set_facecolor("#FF9800")
axes[1].set_xticklabels(["Legitimate", "Fraud"])
axes[1].set_ylabel("Transaction Amount ($)")
axes[1].set_title("Amount Box Plot")
axes[1].grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Legitimate - Mean: ${legit_amounts.mean():.2f}, Median: ${legit_amounts.median():.2f}, Max: ${legit_amounts.max():.2f}")
print(f"Fraud      - Mean: ${fraud_amounts.mean():.2f}, Median: ${fraud_amounts.median():.2f}, Max: ${fraud_amounts.max():.2f}")
print("Fraudulent transactions tend to have smaller amounts on average but with higher variance.")

## Feature Engineering

In [ ]:
# Create time-based features
df["Hour"] = (df["Time"] / 3600) % 24
df["Hour_sin"] = np.sin(2 * np.pi * df["Hour"] / 24)
df["Hour_cos"] = np.cos(2 * np.pi * df["Hour"] / 24)
df = df.drop(columns=["Time"])
print(f"Shape after time features: {df.shape}")
print("Added: Hour, Hour_sin, Hour_cos; Dropped: Time")

In [ ]:
# Separate features/target and stratified split
X = df.drop(columns=["Class"])
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]:,} samples ({y_train.mean()*100:.4f}% fraud)")
print(f"Test:  {X_test.shape[0]:,} samples ({y_test.mean()*100:.4f}% fraud)")
print("Stratified split preserved class balance across sets")

In [ ]:
# Scale Amount AFTER split (no data leakage)
scaler = StandardScaler()
X_train["Amount_scaled"] = scaler.fit_transform(X_train[["Amount"]])
X_test["Amount_scaled"] = scaler.transform(X_test[["Amount"]])
X_train = X_train.drop(columns=["Amount"])
X_test = X_test.drop(columns=["Amount"])
print(f"Training features: {X_train.shape}")
print(f"Test features:     {X_test.shape}")
print("Amount scaled on training set, transformed on test set")

## Modeling

Training 3 models with default parameters — no hyperparameter tuning.
Using `class_weight='balanced'` for LR/RF and `scale_pos_weight` for XGBoost
to handle class imbalance.

In [ ]:
# Define models
scale_pos_weight_val = len(y_train[y_train == 0]) / len(y_train[y_train == 1])
models = {
    "Logistic Regression": LogisticRegression(
        class_weight="balanced", max_iter=1000, random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100, class_weight="balanced", random_state=42, n_jobs=-1
    ),
    "XGBoost": xgb.XGBClassifier(
        n_estimators=100, scale_pos_weight=scale_pos_weight_val,
        eval_metric="logloss", random_state=42, n_jobs=-1
    ),
}
print(f"scale_pos_weight for XGBoost: {scale_pos_weight_val:.2f}")
print(f"Models defined: {', '.join(models.keys())}")

In [ ]:
# Train all models and predict
results = {}
for name, model in models.items():
    print(f"Training {name}...", end=" ")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    results[name] = {"pred": y_pred, "prob": y_prob}
    print("\u2713")
print("\nAll models trained successfully")

In [ ]:
# Evaluate: PR-AUC, Precision, Recall, F1
metrics_data = []
for name, res in results.items():
    pr_auc = average_precision_score(y_test, res["prob"])
    precision = precision_score(y_test, res["pred"])
    recall = recall_score(y_test, res["pred"])
    f1 = f1_score(y_test, res["pred"])
    metrics_data.append({
        "Model": name,
        "PR-AUC": round(pr_auc, 4),
        "Precision": round(precision, 4),
        "Recall": round(recall, 4),
        "F1 Score": round(f1, 4),
    })
metrics_df = pd.DataFrame(metrics_data)
metrics_df

In [ ]:
# Confusion Matrices for all 3 models
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for idx, (name, res) in enumerate(results.items()):
    cm = confusion_matrix(y_test, res["pred"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[idx],
                xticklabels=["Legitimate", "Fraud"],
                yticklabels=["Legitimate", "Fraud"],
                cbar=False, annot_kws={"fontsize": 12})
    axes[idx].set_title(name, fontsize=12, fontweight="bold")
    axes[idx].set_ylabel("Actual")
    axes[idx].set_xlabel("Predicted")
fig.suptitle("Confusion Matrices", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## Model Comparison

In [ ]:
# Comparison table sorted by PR-AUC
metrics_df_sorted = metrics_df.sort_values("PR-AUC", ascending=False).reset_index(drop=True)
metrics_df_sorted

### Best Model Selection

XGBoost and Random Forest perform similarly (PR-AUC 0.819 vs 0.815), both
significantly beating Logistic Regression (0.68).

I'd choose **Random Forest** for production because:
- **Higher precision (0.959 vs 0.913)** — fewer false alarms, which matters for fraud teams
- **Robust to outliers** — ensemble of trees handles noisy PCA features well
- **Interpretable** — provides feature importance for understanding what drives fraud predictions

## Feature Importance

In [ ]:
# Feature importance from Random Forest
rf_model = models["Random Forest"]
importance = rf_model.feature_importances_
feature_names = X_train.columns
importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importance
}).sort_values("importance", ascending=True).tail(15)

plt.figure(figsize=(10, 6))
colors = plt.cm.Blues(np.linspace(0.3, 0.9, len(importance_df)))
plt.barh(range(len(importance_df)), importance_df["importance"].values, color=colors, edgecolor="white")
plt.yticks(range(len(importance_df)), importance_df["feature"].values)
plt.xlabel("Importance")
plt.title("Top 15 Features by Importance — Random Forest", fontsize=14, fontweight="bold")
plt.grid(True, axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

### Feature Interpretation

**V14 is the most important feature** for detecting fraud, followed by V17, V10, V12, and V11.
These PCA-transformed features capture hidden patterns in the original transaction data that
strongly distinguish fraud from legitimate transactions.

The time-based features (Hour_sin, Hour_cos, Hour) rank lower, suggesting that
transaction timing is less discriminative than the PCA components for this dataset.
Amount_scaled also contributes meaningfully but is not among the top drivers.

## Key Findings

1. **Fraud is rare but detectable**: With only 0.17% fraud rate, our best models achieve PR-AUC of 0.82, catching ~75-80% of fraud cases with high precision.

2. **PCA features carry the signal**: Although the original features are PCA-transformed for confidentiality, V14, V17, V10, and V12 retain strong discriminatory power between fraud and legitimate transactions.

3. **Tree-based models outperform linear baselines**: Random Forest (PR-AUC 0.815) and XGBoost (0.819) significantly beat Logistic Regression (0.68), showing that fraud patterns involve non-linear interactions between features that linear models cannot capture.